# Weighted CFG-Ablation Aggregate Forest Plot

This notebook combines the prompt-matched, prompt-mismatched, and out-of-range CFG ablations into one $3\times4$ forest plot. The rows follow that experiment order, and one shared CFG legend appears above the full figure.

In [ ]:
from pathlib import Path
import importlib
import importlib.util

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / 'analyze_results' / 'sd15_recovery_analysis.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the ActiveConditionalGCS project root.')

module_path = PROJECT_ROOT / 'analyze_results' / 'weighted' / 'ablation' / 'analysis.py'
module_spec = importlib.util.spec_from_file_location('weighted_ablation_analysis', module_path)
if module_spec is None or module_spec.loader is None:
    raise ImportError(f'Could not load {module_path}.')
diagnostic = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(diagnostic)
diagnostic.recovery = importlib.reload(diagnostic.recovery)

SCENARIOS = ('prompt_matched', 'prompt_mismatched', 'out_of_range')
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'weighted' / 'ablation' / 'summary' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for scenario in SCENARIOS:
    diagnostic.ensure_lpips(scenario, device='cpu')
FRAMES = [diagnostic.load_rows(scenario) for scenario in SCENARIOS]
for scenario, frame in zip(SCENARIOS, FRAMES):
    print(f'{scenario}: {len(frame)} / {diagnostic.EXPECTED_ROWS_PER_SCENARIO} rows')

## Combined Aggregate Metrics

From top to bottom, the rows show the in-range prompt-matched, in-range prompt-mismatched, and out-of-range CFG ablations. Metric names appear only beneath the bottom row.

In [ ]:
FOREST_OUTPUT = diagnostic.plot_combined_aggregate_forest(
    FRAMES,
    output_path=OUTPUT_DIR / 'aggregate_cfg_forest_combined.pdf',
    show=True,
)
FOREST_OUTPUT